<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%204/Aula_4_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4.3 — Workflows com lógica avançada + roteamento

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 4 — Orquestrando agentes com workflows

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 4.2** construímos o workflow completo:
- `Parallel` na coleta,
- Auxiliar Técnico -> `Escalacao` tipada,
- Sequencial

Nesta aula, três evoluções:

1. **`Loop`**
2. **`Condition`**
3. **Roteador time vs workflow**


## Setup


In [ ]:
!pip install -U agno openai tavily-python wikipedia pydantic

In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 0 — Recriando os agentes da 4.2


In [3]:
from pydantic import BaseModel, Field
from typing import List

class Jogador(BaseModel):
    nome: str = Field(..., description="Nome do jogador")
    posicao: str = Field(..., description="Posição em campo")
    motivo: str = Field(..., description="Justificativa para a escalação")

class Escalacao(BaseModel):
    formacao: str = Field(..., description="Formação tática (ex: 4-3-3)")
    titulares: List[Jogador] = Field(..., description="11 jogadores titulares")
    reservas: List[Jogador] = Field(..., description="3-5 reservas-chave")
    capitao: str = Field(..., description="Nome do capitão")
    estrategia_resumida: str = Field(..., description="Estratégia em 2-3 frases")

In [4]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_olheiro   = OpenAIChat(id="gpt-5.4")
modelo_analista  = OpenAIChat(id="gpt-5.4")
modelo_auxiliar  = OpenAIChat(id="gpt-5.4")
modelo_treinador = OpenAIChat(id="gpt-5.4")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas e em base interna oficial",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável.",
        "Você tem duas fontes disponíveis:",
        "• Tavily (web): eventos recentes — últimos jogos, convocações, forma de jogadores.",
        "• Wikipedia: fatos históricos consolidados — Copas antigas, biografias, técnicos do passado.",
        "Para forma atual, use Tavily. Para histórico, use Wikipedia. Combine fontes quando a pergunta tiver múltiplas necessidades.",
        "Retorne dados objetivos — não interprete nem opine.",
        "Se a busca falhar em todas as fontes, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

analista = Agent(
    name="Analista",
    role="Interpreta dados e identifica padrões táticos",
    model=modelo_analista,
    instructions=[
        "Você é o Analista de desempenho do ScoutAI FC.",
        "Recebe dados crus e produz leitura tática estruturada.",
        "Identifique: forma atual dos jogadores, vulnerabilidades do adversário, "
        "padrões táticos relevantes.",
        "Devolva análise objetiva, sem ainda sugerir escalação.",
    ],
    markdown=True,
)

treinador = Agent(
    name="Treinador do ScoutAI FC ",
    role="Sintetiza dados e produz recomendação tática para o usuário",
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC — uma plataforma de IA dedicada à Seleção Brasileira masculina de futebol.",
        "Sua especialidade é a Seleção: história, conquistas, jogadores, técnicos, táticas e cultura ao redor do time.",
        "Seu público são torcedores, analistas e comissões técnicas. Adapte o nível técnico ao tipo de pergunta.",

        "Sua função: produzir uma recomendação de escalação clara e estruturada.",
        "Sempre inclua: formação sugerida (ex: 4-3-3), 11 titulares com posição, "
        "principais reservas e justificativa tática em 2-3 frases.",
        "Responda em português do Brasil, com tom profissional.",
    ],
    markdown=True,
)

---

## Passo 1 — `Loop`: refinando a análise até qualidade aceitável

- Análise muito variável (Analista)  
- Garantia de um padrão mínimo
-  A solução: **`Loop`**
- Critério desta aula: **300 caracteres**


In [5]:
def analise_aceitavel(outputs) -> bool:
    """Critério de saída do Loop: análise aceitável tem pelo menos 300 caracteres.

    `outputs` é a lista de outputs da iteração. Retornar True sai do loop.
    """
    if not outputs:
        return False
    ultima_analise = outputs[-1].content or ""
    return len(str(ultima_analise)) >= 300

---

## Passo 2 — `Condition` com `else_steps`: bifurcação real

A solução: **`Condition`**. Ele avalia uma função e:
- Se `True` → executa `steps`
- Se `False` → executa `else_steps` (caminho alternativo)

Critério didático (termos):

"formação", "esquema", "marcação", "pressão"


In [6]:
def precisa_pesquisa_extra(step_input) -> bool:
    """Indica se a análise precisa de complemento sobre o adversário.

    `step_input.previous_step_content` é o output do step anterior.
    Critério: se análise não menciona termos táticos do adversário, precisa de mais pesquisa.
    """
    analise = str(step_input.previous_step_content or "").lower()
    palavras_adversario = ["formação", "esquema", "marcação", "pressão"]
    return not any(p in analise for p in palavras_adversario)

from agno.workflow import StepOutput
# Step de pesquisa complementar (executa o Olheiro com instrução adicional)
def nota_analise_completa(step_input):
    """Step que apenas adiciona uma nota — caminho alternativo quando análise já está completa."""
    return StepOutput(
        content=str(step_input.previous_step_content or "") +
                "\n\n[Análise já contemplava aspectos táticos do adversário — pesquisa extra dispensada.]"
    )

---

## Passo 3 — Auxiliar Técnico



In [7]:
auxiliar_tecnico = Agent(
    name="Auxiliar Técnico",
    role="Decide a escalação ideal com base em dados e análise tática",
    model=modelo_auxiliar,
    instructions=[
        "Você é o Auxiliar Técnico do ScoutAI FC, especialista em montar escalação.",
        "Recebe dados sobre os jogadores e análise tática do adversário.",
        "Sua função: decidir formação, 11 titulares, 3-5 reservas, capitão e estratégia.",
        "Sempre justifique cada escolha de jogador no campo 'motivo'.",
        "Mantenha coerência entre a formação escolhida e os jogadores titulares.",
        "Resolva problemas passo a passo antes de responder.",
        "Explique seu raciocínio de forma estruturada.",
    ],
    output_schema=Escalacao,
    markdown=True,
)

---

## Passo 4 — Montando o workflow robusto

Aqui combinamos todas as peças:
- `Parallel`,
- `Loop`,
- `Condition`.

```
Parallel (coleta dual)
   ├── Coleta jogadores
   └── Coleta adversário
        │
        ▼
Loop (até análise atingir 300+ chars)  [max_iterations=2]
   └── Análise tática
        │
        ▼
Condition (precisa pesquisa extra do adversário?)
   ├── True → Pesquisa complementar (Olheiro)
   └── False → Nota de análise completa
        │
        ▼
Step: Decisão (Auxiliar Técnico) → Escalacao
        │
        ▼
Step: Apresentação (Treinador)
```


In [8]:
from agno.workflow import Step, Workflow, Parallel, Loop, Condition

workflow_robusto = Workflow(
    name="Recomendação de Escalação Robusta",
    steps=[
        # Etapa 1: coleta dual em paralelo
        Parallel(
            Step(name="Coleta jogadores", agent=olheiro),
            Step(name="Coleta adversário", agent=olheiro),
        ),

        # Etapa 2: análise com refinamento iterativo
        Loop(
            name="Refinar análise até qualidade aceitável",
            steps=[Step(name="Análise", agent=analista)],
            end_condition=analise_aceitavel,
            max_iterations=2,
        ),

        # Etapa 3: pesquisa complementar condicional
        Condition(
            name="Pesquisa extra se necessário",
            evaluator=precisa_pesquisa_extra,
            steps=[Step(name="Pesquisa complementar", agent=olheiro)],
            else_steps=[Step(name="Análise já completa", executor=nota_analise_completa)],
        ),

        # Etapa 4: decisão da escalação (com reasoning)
        Step(name="Decisão", agent=auxiliar_tecnico),

        # Etapa 5: apresentação ao usuário
        Step(name="Apresentação", agent=treinador),
    ],
)

---

## Passo 5 — Workflow robusto em ação



In [9]:
workflow_robusto.print_response(
   "Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), "
    "considerando forma física atual dos convocados e o adversário. Exclua da lista jogadores lesionados.",
    stream=True,
)

┏━ Workflow Information ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Workflow: Recomendação de Escalação Robusta                                                                     ┃
┃                                                                                                                 ┃
┃ Steps: 5 steps                                                                                                  ┃
┃                                                                                                                 ┃
┃ Message: Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), considerando forma física  ┃
┃ atual dos convocados e o adversário. Exclua da lista jogadores lesionados.                                      ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Output()

┏━ Step 1: Parallel Steps ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Parallel Steps: 2                                                                                               ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

INFO Searching wikipedia for: Brazil national football team Panama national football team

ERROR    Error searching Wikipedia for 'Brazil national football team Panama national football team': Expecting    
         value: line 1 column 1 (char 0)

...

┏━ Loop Refinar análise até qualidade aceitável (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Loop Summary:                                                                                                   ┃
┃                                                                                                                 ┃
┃  • Total iterations: 1/2                                                                                        ┃
┃  • Total steps executed: 1                                                                                      ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

┏━ Step 3.1: Pesquisa complementar (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃  Análise tática estruturada: Brasil x PanamáA coleta apresentada organiza de forma clara os pontos centrais do  ┃
┃    confronto e permite uma leitura objetiva sobre estado atual dos jogadores citados, estrutura provável das    ┃
┃                  equipes e vulnerabilidades observáveis, sem entrar em proposta de escalação.                   ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃ ##1. Estado atual do elenco brasileiro### Disponibilidade e contexto físicoO recorte indica que o Brasil        ┃
┃ trabalhou com foco em jogadores considerados disponíveis e em condição competitiva mais segura, enquanto        ┃
┃ algumas ausências recentes afetaram a composição do grupo.                                                      ┃
┃                                                                                                                 ┃
┃                                        Baixas citadas na coleta- Alisson                                        ┃
┃                                                                                                                 ┃
┃  • Alex Sandro                                                                                                  ┃
┃  • Bruno Guimarães                                                                                              ┃
┃  • Estevão                                                                                                      ┃
┃  • Rodrygo                                                                                                      ┃
┃  • Éder Militão aparece com informação contraditória: em um ponto como lesionado, em outro como retornando após ┃
┃    recuperação                                                                                                  ┃
┃  • Neymar não aparece no recorte recente### Efeito estrutural das ausências- Bruno Guimarães: perda de          ┃
┃    circulação interna, apoio na saída e controle de ritmo.                                                      ┃
┃  • Rodrygo: reduz mobilidade associativa e conexão do lado direito para dentro.                                 ┃
┃  • Alisson: impacto na construção curta e no controle da área.                                                  ┃
┃  • Alex Sandro: perda de estabilidade e experiência no lado esquerdo.                                           ┃
┃  • Estevão: menos profundidade e desequilíbrio individual.                                                      ┃
┃  • Neymar fora do recorte: reduz centralidade criativa por dentro.                                              ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃ ##2. Jogadores brasileiros em evidência## Bloco defensivo### Danilo                                             ┃
┃                                                                                                                 ┃
┃  • Nome de experiência e recorrência.                                                                           ┃
┃  • Perfil associado a equilíbrio posicional, apoio à saída e controle do setor.                                 ┃
┃                                                                                                                 ┃
┃                         Bremer / Marquinhos / Roger Ib

┏━ Step 4: Decisão (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Structured Output:                                                                                              ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃  {                                                                                                              ┃
┃    "formacao": "4-3-3",                                                                                         ┃
┃    "titulares": [                                                                                               ┃
┃      {                                                                                                          ┃
┃        "nome": "Bento",                                                                                         ┃
┃        "posicao": "Goleiro",                                                                                    ┃
┃        "motivo": "Com Alisson fora, oferece segurança básica de área e perfil adequado para sustentar um time   ┃
┃  que terá posse alta, mantendo atenção à bola longa e às transições do Panamá."                                 ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Danilo",                                                                                        ┃
┃        "posicao": "Lateral-direito",                                                                            ┃
┃        "motivo": "É a peça mais confiável para dar equilíbrio posicional no lado direito, importante para       ┃
┃  controlar as costas do setor contra as transições de Ismael Díaz ou José Luis Rodríguez."                      ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Marquinhos",                                                                                    ┃
┃        "posicao": "Zagueiro",                                                                                   ┃
┃        "motivo": "Combina liderança, leitura de cobertura e qualidade no passe para organizar a última linha e  ┃
┃  ajudar na saída contra pressão inicial ou bolas diretas."                                                      ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Bremer",                                                                                        ┃
┃        "posicao": "Zagueiro",                                                                                   ┃
┃        "motivo": "Entra pela força nos duelos, proteção de área e resposta ao jogo físico de Waterman, além de  ┃
┃  ser útil nas segundas bolas."                                                                                  ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Éder Militão",                                                                                  ┃
┃        "posicao": "Lateral-esquerdo",                 

┏━ Step 5: Apresentação (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                                     Escalação recomendada — Brasil em 4-3-3                                     ┃
┃                                                                                                                 ┃
┃                                                  Time titular                                                   ┃
┃                                                                                                                 ┃
┃  1 Bento — Goleiro2. Danilo — Lateral-direito3. Marquinhos — Zagueiro4. Bremer — Zagueiro5. Éder Militão —      ┃
┃    Lateral-esquerdo6. Casemiro (capitão) — Volante7. Andrey Santos — Meia8. Gabriel Sara — Meia9. Luiz Henrique ┃
┃    — Ponta-direita10. Matheus Cunha — Centroavante11. Vinícius Júnior — Ponta-esquerda---                       ┃
┃                                                                                                                 ┃
┃                                               Principais reservas                                               ┃
┃                                                                                                                 ┃
┃  • Endrick — Atacante- Gabriel Martinelli — Ponta- João Pedro — Atacante- Roger Ibañez — Zagueiro- Igor Thiago  ┃
┃    — Centroavante---                                                                                            ┃
┃                                                                                                                 ┃
┃                                              Justificativa tática                                               ┃
┃                                                                                                                 ┃
┃ A proposta faz sentido para um jogo em que o Brasil deve ter mais posse, empurrar o Panamá para trás e atacar   ┃
┃ bloco médio/baixo com amplitude e1x1 pelos lados. Vinícius Júnior e Luiz Henrique alargam o campo, enquanto     ┃
┃ Gabriel Sara oferece conexão por dentro e Matheus Cunha dá mobilidade para tirar os zagueiros da zona de        ┃
┃ conforto.                                                                                                       ┃
┃                                                                                                                 ┃
┃ Sem bola, a base da equipe está no equilíbrio de Casemiro, na cobertura de Danilo e na força física de          ┃
┃ Marquinhos + Bremer para controlar transições e bolas diretas. A utilização de Militão na esquerda deixa o time ┃
┃ mais conservador defensivamente, priorizando proteção do corredor e recuperação rápida caso o Panamá tente      ┃
┃ acelerar pelos lados.                                                                                           ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                           Leitura tática complementar                                           ┃
┃                                                                                                                 ┃
┃  • Ponto forte: superioridade técnica nas pontas e controle das transições.                                     ┃
┃  • Risco: menor profundidade ofensiva pelo lado esquerdo com Militão, o que pode concentrar criação excessiva   ┃
┃    em Vinícius.                                                                                                 ┃
┃  • Ajuste de jogo: se o Brasil precisar aumentar o vol

Completed in 162.1s

---

## Passo 6 — Roteador entre time e workflow

- **Time conversacional** (Aula 3) — boa pra perguntas exploratórias, dúvidas, comparações
- **Workflow estruturado** (esta aula) — bom pra decisões repetidas, escalação, recomendações estruturadas



In [10]:
class DecisaoRoteamento(BaseModel):
    mecanismo: str = Field(..., description="Mecanismo escolhido: 'workflow' ou 'time'")
    motivo: str = Field(..., description="Breve justificativa")

modelo_roteador  = OpenAIChat(id="gpt-5.4")
roteador = Agent(
    name="Roteador",
    role="Classifica a pergunta do usuário e decide qual mecanismo deve respondê-la",
    model=modelo_roteador,
    instructions=[
        "Classifique a pergunta do usuário em 'workflow' ou 'time'.",
        "'workflow' — quando a pergunta pede DECISÃO ESTRUTURADA REPETÍVEL: "
        "  recomendar escalação, montar time, sugerir formação titular, "
        "  ou qualquer pergunta cujo formato de resposta deva ser sempre o mesmo.",
        "'time' — quando a pergunta é EXPLORATÓRIA, CONVERSACIONAL ou ANALÍTICA: "
        "  comparar jogadores, explicar conceito tático, discutir histórico, "
        "  análise livre, opinião sobre adversário.",
        "Responda APENAS com a classificação no formato pedido.",
    ],
    output_schema=DecisaoRoteamento,
    markdown=False,
)

### Construindo a função `responder()` com roteamento

Pra usar o roteador, precisamos do **time da Aula 3** disponível também. Vamos recriar a versão enxuta (Olheiro como member, Treinador-líder).


In [15]:
from agno.team import Team

time_scoutai = Team(
    name="Treinador do ScoutAI FC",
    mode="coordinate",
    members=[olheiro],
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC, líder da comissão técnica.",
        "Delegue ao Olheiro quando precisar de dados externos verificáveis.",
        "Para perguntas conceituais simples, responda direto sem delegar.",
        "Integre os dados com sua análise tática antes de responder em português do Brasil.",
    ],
    markdown=True,
)


def responder(mensagem: str, historico=None) -> str:
    """Função do fio condutor — agora com roteamento entre time e workflow.

    1. Roteador classifica a pergunta
    2. Despacha pro mecanismo escolhido
    3. Devolve a resposta final
    """
    # 1. Classificação
    decisao = roteador.run(mensagem).content
    print(f"[Roteador] mecanismo={decisao.mecanismo} | motivo={decisao.motivo}")

    # 2. Despacho
    if decisao.mecanismo == "workflow":
        resposta = workflow_robusto.run(mensagem)
    else:
        resposta = time_scoutai.run(mensagem)

    return resposta.content

### Testando o roteador

Duas perguntas de naturezas diferentes — esperado: roteador escolhe mecanismos distintos.


In [16]:
# Pergunta 1: pede DECISÃO ESTRUTURADA → deve ir pro workflow
print(">>> Pergunta 1: recomendação de escalação\n")
resposta = responder("Recomenda a escalação ideal da Seleção pro próximo jogo.")
print(resposta[:500] + "..." if len(resposta) > 500 else resposta)

>>> Pergunta 1: recomendação de escalação

[Roteador] mecanismo=workflow | motivo=O pedido é para recomendar uma escalação ideal para o próximo jogo, uma decisão estruturada e repetível.
## Escalação recomendada da Seleção Brasileira

### Formação sugerida: **4-2-3-1**

### 11 titulares
1. **Bento** — GOL  
2. **Wesley** — LD  
3. **Éder Militão** — ZAG  
4. **Marquinhos** — ZAG **(capitão)**  
5. **Caio Henrique** — LE  
6. **Casemiro** — VOL  
7. **Bruno Guimarães** — VOL  
8. **Estêvão** — MEI/PD  
9. **Rodrygo** — MEI/SA  
10. **Vinícius Júnior** — PE  
11. **Matheus Cunha** — CA  

### Principais reservas
- **Ederson** — GOL  
- **Alex Sandro** — LE  
- **Gabriel Magalhães*...


In [17]:
# Pergunta 2: CONVERSACIONAL → deve ir pro time
print(">>> Pergunta 2: explicação tática\n")
resposta = responder("Quais as vantagens do esquema 3-5-2 contra times que jogam com 4-4-2?")
print(resposta[:500] + "..." if len(resposta) > 500 else resposta)

>>> Pergunta 2: explicação tática

[Roteador] mecanismo=time | motivo=Pergunta analítica e exploratória sobre conceito tático e comparação entre esquemas, não pede uma decisão estruturada repetível.
## Vantagens do **3-5-2** contra um **4-4-2**

Contra um time em **4-4-2**, o **3-5-2** costuma oferecer vantagens bem claras, principalmente em **superioridade no meio-campo**, **amplitude com alas** e **coberturas defensivas**.

### 1. **Superioridade numérica no meio**
No 4-4-2 tradicional, o adversário costuma ter **2 meio-campistas centrais**.  
No 3-5-2, você normalmente atua com **3 jogadores por dentro** no meio.

**Vantagem prática:**
- mais controle da posse;
- mais linhas de passe;
- ...


---

### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Time conversacional (Aula 3) — perguntas exploratórias
├── ✅ Workflow estruturado robusto (esta aula)
│   ├── Parallel: coleta dual
│   ├── Loop: refinar análise (max 2x)
│   ├── Condition: pesquisa extra com else_steps
│   ├── Step: decisão com reasoning + Pydantic
│   └── Step: apresentação
├── ✅ Roteador inteligente entre os dois mecanismos
│
└── ❌ Sem produção / monitoramento / governança / persistência → Aula 5 (AgentOS)
```

### Aplicando em outros contextos

| Domínio | Time conversacional | Workflow estruturado |
|---|---|---|
| **Atendimento ao cliente** | "qual a política de troca?" | "abrir ticket de reembolso" |
| **Análise financeira** | "explica esse indicador" | "gerar relatório de risco do portfólio" |
| **Jurídico** | "explicar precedente" | "gerar minuta de contrato" |

<br>

### Próxima aula

**Aula 5 — Gerenciando agentes com AgentOS**

